# Binning and Discretization

**Topic:** Feature Engineering

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import Dropdown, IntSlider, FloatSlider, Output, HBox, VBox
from IPython.display import display, clear_output, HTML
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** the difference between equal-width and equal-frequency binning
- **Explain** when binning a continuous feature helps a model versus when it hurts
- **Interpret** a distribution before and after binning and recognize how they handle skew differently

> **Tip:** Toggle between equal-width and equal-frequency binning on the Airbnb price chart below. On the left, watch where the dotted bin-edge lines land against the raw price histogram. On the right, watch how evenly (or unevenly) listings spread across the bins.

---
## How we got here

In **[02_creating_new_features.ipynb](02_creating_new_features.ipynb)** you created new features by combining raw columns. Sometimes, though, the right move is the opposite: simplify a noisy continuous feature by grouping values into discrete buckets.

Binning is the technique for that, and it comes in two flavors with very different behavior on skewed data.

---
## Why this matters for data science

Not every model benefits from knowing that a price is \$127 versus \$131. Tree-based models find splits on their own, but linear models and neural networks can be thrown off by a long tail of outliers. Binning converts that noisy continuous value into a stable group: "Budget," "Mid-range," "Premium," "Luxury."

The tradeoff is real: you lose precision to gain stability. Knowing when that trade is worth making is a judgment call that separates thoughtful feature engineers from mechanical ones.

---
## Try it yourself

In [2]:
_airbnb_w3 = pd.read_csv('../../data/nyc_airbnb.csv')
_airbnb_w3 = _airbnb_w3[_airbnb_w3['price'] > 0].copy()

out_bin = Output()
_initialized = False

nbins_sl_w3 = IntSlider(value=10, min=5, max=20, step=1,
    description="Bins:", style={"description_width": "60px"},
    layout=widgets.Layout(width="380px"))
bin_method_dd_w3 = Dropdown(options=["Equal-width (pd.cut)", "Equal-frequency (pd.qcut)"],
    value="Equal-width (pd.cut)", description="Method:",
    style={"description_width": "70px"}, layout=widgets.Layout(width="320px"))

def render_bin(change=None):
    global _initialized
    n = nbins_sl_w3.value
    method = bin_method_dd_w3.value
    if method.startswith("Equal-width"):
        bins = pd.cut(_airbnb_w3['price'], bins=n)
    else:
        bins = pd.qcut(_airbnb_w3['price'], q=n, duplicates="drop")

    bin_counts = bins.value_counts().sort_index()
    n_actual = len(bin_counts)
    bin_labels = [f"${iv.mid:,.0f}" for iv in bin_counts.index]
    edges = [iv.right for iv in bin_counts.index[:-1]]

    first_pct = bin_counts.iloc[0] / len(_airbnb_w3) * 100
    n_empty = int((bin_counts == 0).sum())

    fig = make_subplots(rows=1, cols=2, subplot_titles=(
        "Raw price, with bin edges marked", "Listings per bin"))

    fig.add_trace(go.Histogram(
        x=_airbnb_w3['price'], nbinsx=60,
        marker_color=PALETTE["primary"], opacity=0.75,
    ), row=1, col=1)
    for edge in edges:
        fig.add_vline(x=edge, line=dict(color=PALETTE["muted"], width=1, dash="dot"),
            row=1, col=1)

    fig.add_trace(go.Bar(x=bin_labels, y=bin_counts.values,
        marker_color=PALETTE["secondary"]), row=1, col=2)

    fig.update_xaxes(title_text="Price ($)", row=1, col=1)
    fig.update_xaxes(title_text="Bin (labeled by midpoint)", tickangle=45, row=1, col=2)
    fig.update_yaxes(title_text="Listings", row=1, col=1)
    fig.update_yaxes(title_text="Listings", row=1, col=2)
    fig.update_layout(
        height=440, showlegend=False,
        paper_bgcolor=PALETTE["background"], plot_bgcolor=PALETTE["surface"],
        font=dict(family=FONT["family"]),
    )

    if method.startswith("Equal-width"):
        caption_body = (
            f"With {n} equal-width bins, {first_pct:.1f}% of listings land in the first bin "
            f"and {n_empty} bin{'s' if n_empty != 1 else ''} on the right "
            f"{'are' if n_empty != 1 else 'is'} completely empty. Look at the left panel: "
            f"most of the dotted bin-edge lines sit out in the sparse, near-empty stretch to "
            f"the right of where the bulk of the listings actually are."
        )
    else:
        avg_share = 100 / n_actual
        caption_body = (
            f"With {n_actual} equal-frequency bins, every bin holds about {avg_share:.1f}% of "
            f"listings and none are empty. But look at the left panel: the dotted bin-edge "
            f"lines are packed tightly together on the left, where most listings are priced, "
            f"and spread far apart on the right, where listings are sparse. Equal price ranges "
            f"were traded for equal listing counts."
        )

    caption = (
        f'<div style="font-family:{FONT["family"]};font-size:13px;background:#f8f9fa;'
        f'padding:12px 16px;border-radius:6px;margin-top:8px;line-height:1.6;color:#333;">'
        f'<p style="margin:0;">{caption_body}</p></div>'
    )

    with out_bin:
        clear_output(wait=True)
        fig.show()
        display(HTML(caption))
    _initialized = True

nbins_sl_w3.observe(render_bin, names="value")
bin_method_dd_w3.observe(render_bin, names="value")
display(VBox([nbins_sl_w3, bin_method_dd_w3, out_bin]))
render_bin()

---
## What's happening?

**Equal-width binning** (`pd.cut`) divides the range of values into N intervals of equal size. Airbnb prices run from \$10 to \$3,706, so with 5 bins each one covers about \$739. Because price is right-skewed (a long tail of expensive listings pulls the range out much further than most prices actually reach), almost every listing lands in the first bin and the rest sit nearly empty.

**Equal-frequency binning** (`pd.qcut`) divides values so each bin contains roughly the same number of observations. For 5 bins, each contains ~20% of listings. This spreads the data more evenly but means the bins cover very different dollar ranges — the first bin might span \$10–\$69, while the last spans hundreds of dollars.

```python
# Equal-width
airbnb['price_bin_ew'] = pd.cut(airbnb['price'], bins=5,
                                 labels=['Budget','Low','Mid','High','Luxury'])

# Equal-frequency
airbnb['price_bin_ef'] = pd.qcut(airbnb['price'], q=5, duplicates='drop',
                                  labels=['Q1','Q2','Q3','Q4','Q5'])
```

`duplicates='drop'` guards against a `qcut` error: if a column has many repeated values, some quantile edges can come out identical, and `qcut` refuses to build duplicate bins unless told to drop them.

| Binning type | How it works | When to use | Risk |
|-------------|-------------|-------------|------|
| Equal-width (`pd.cut`) | Same value range per bin | Uniform distributions | Empty bins on skewed data |
| Equal-frequency (`pd.qcut`) | Same count per bin | Skewed distributions | Unequal value ranges |
| Custom thresholds | You define the edges | Domain-driven groups (Budget / Mid / Luxury) | Requires domain knowledge |
| KMeans binning | Cluster-based boundaries | Non-obvious groupings | Computationally heavier |

---
## Real-world example: Binning Airbnb Price into Budget Tiers

The chart below applies equal-frequency binning to Airbnb price, creating five tiers from Budget to Luxury, then shows the count of listings in each tier by neighbourhood group.

In [3]:
np.random.seed(42)
airbnb = pd.read_csv('../../data/nyc_airbnb.csv')
airbnb = airbnb[airbnb['price'] > 0].copy()

airbnb['price_tier'] = pd.qcut(
    airbnb['price'], q=5, duplicates='drop',
    labels=['Budget', 'Economy', 'Mid-range', 'Premium', 'Luxury']
)

tier_ng = (airbnb.groupby(['price_tier', 'neighbourhood_group'], observed=True)
                 .size().reset_index(name='count'))

fig = go.Figure()
colors = [PALETTE['primary'], PALETTE['secondary'], PALETTE['accent'],
          PALETTE['muted'], '#AE3EC9']
for i, ng in enumerate(['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island']):
    sub = tier_ng[tier_ng['neighbourhood_group'] == ng]
    fig.add_trace(go.Bar(
        name=ng, x=sub['price_tier'].astype(str), y=sub['count'],
        marker_color=colors[i],
    ))
layout = base_layout(
    title='Airbnb Listings by Price Tier and Neighbourhood Group',
    xaxis_title='Price tier (equal-frequency bins)',
    yaxis_title='Number of listings',
)
layout.update(barmode='group')
go.Figure(data=fig.data, layout=layout).show()

> **Discussion question:** Under equal-width binning, nearly all listings would fall into "Budget." What does that tell you about the Airbnb price distribution, and which binning method is more useful here?

---
## Key takeaway

> **Binning trades precision for stability — use it when a continuous feature is noisy or right-skewed and the model benefits more from knowing the group than the exact value.**

---
*Next up: 04_interaction_features — where two features combine to capture what neither tells you alone*